## Hent data

- BVP dataframes har 64 observationer/sek
- EDA dataframes har 4 observationer/sek
- HR dataframes har 1 observationer/sek
- TEMP dataframes har 4 observationer/sek

### Vi vælger at interpollere og downscale til 16 punkter i sekundet for at beholde mere information fra BVP

- BVP: 64 -> 16 (mean på hvert 4 punkt)
- EDA: 4 -> 16 (interpoler 3 nye punkter mellem hvert punktsæt)
- HR: 1 -> 16 (interpoler 15 nye punkter mellem hvert sekund)
- TEMP: 4 -> 16 (interpoler 15 nye punkter mellem hvert punktsæt)

In [44]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

In [29]:
def prepare_dataframe(path_to_phase, file):
    colname = file.split('.')[0]
    df = pd.read_csv(path_to_phase+'/'+ file)
    df['time'] = pd.to_datetime(df['time'], format='mixed')
    df = df.set_index('time').sort_index()

    if file== 'BVP.csv':
        df_downsampled = df.resample("62.5ms").mean()
        return df_downsampled[[colname]]
    else: 
        #sæt en interpoleringsfrekvens på 62.5 ms som svarer til 1/16 sekund og interpoler
        df_upsampled = df.resample("62.5ms").asfreq()
        df_upsampled[colname] = df_upsampled[colname].interpolate(method='linear')
        return df_upsampled[[colname]]
    
    
def create_phase_dataframe(path_to_phase):
    filenames = ['BVP.csv', 'EDA.csv', 'HR.csv', 'TEMP.csv']
    dfs_to_combine = []
    for file in filenames:
        df = prepare_dataframe(path_to_phase, file)
        dfs_to_combine.append(df)
    
    Collected_dataframes = pd.concat(dfs_to_combine, axis=1, join='inner')

    # 3. Hvis du gerne vil have 'time' tilbage som en almindelig kolonne til sidst:
    Collected_dataframes = Collected_dataframes.reset_index()
    return Collected_dataframes
        

path_phase = '/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1/round_1/phase1/'

phase_dataframe = create_phase_dataframe(path_phase)
phase_dataframe.tail()

,time,BVP,EDA,HR,TEMP
7148,2021-12-17 16:19:21.750000,-88.9000,0.287070,78.852500,31.450
7149,2021-12-17 16:19:21.812500,-162.2075,0.286749,78.856875,31.455
7150,2021-12-17 16:19:21.875000,-139.4200,0.286429,78.861250,31.460
7151,2021-12-17 16:19:21.937500,-73.7275,0.286108,78.865625,31.465
7152,2021-12-17 16:19:22.000000,-41.0475,0.285788,78.870000,31.470


In [30]:
import pandas as pd
import os

def append_to_csv(df, filename, path_to_folder):
    filepath = os.path.join(path_to_folder,filename )
    # Check if file exists right now
    file_exists = os.path.isfile(filepath)
    
    # mode='a' : Append to the end of the file
    # header=not file_exists : If file exists, header is False. If not, header is True.
    df.to_csv(filepath, mode='a', index=False, header=not file_exists)
    



In [31]:
# person_nummer = 1
# person_sti = '/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1'
# person_ID_and_group = f'Person{str(person_nummer)}_D1_1_ID_1'
# print(person_ID_and_group)
# person_nummer +=1
# filename_person = person_ID_and_group+'.csv'
# for runde in sorted(os.listdir(person_sti)):
#     runde_sti = os.path.join(person_sti, runde)
#     print(runde_sti)
#     if not os.path.isdir(runde_sti): continue
    
#     # 4. Loop: Phaser
#     for phase in sorted(os.listdir(runde_sti)):
#         phase_sti = os.path.join(runde_sti, phase)
#         #print(phase_sti)
#         if not os.path.isdir(phase_sti): continue
#         phase_df = create_phase_dataframe(phase_sti)
#         append_to_csv(phase_df, filename_person)
#     print(f'round {runde} done')

In [35]:
if os.path.exists('../data/preprocessed/Person1_D1_1_ID_1.csv'):
    os.remove('../data/preprocessed/Person1_D1_1_ID_1.csv')

In [47]:
import pandas as pd
import pathlib
import os
def create_collected_datafile_all_participans(dataset_path='../data/raw/dataset'):

    # Definer stien
    #dataset_path = r'/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset'

    path_to_processed_folder = '../data/preprocessed'

    person_nummer = 1
    for gruppe in sorted(os.listdir(dataset_path)):
        gruppe_sti = os.path.join(dataset_path, gruppe)
        if not os.path.isdir(gruppe_sti): continue
        
        # 2. Loop: Personer
        for person in sorted(os.listdir(gruppe_sti)):
            person_sti = os.path.join(gruppe_sti, person)
            person_ID_and_group = f'Person{str(person_nummer)}_{gruppe}_{person}'
            
            filename_person = person_ID_and_group+'.csv'
            # Hvis koden er kørt før skal personfilerne slettes og laves på ny. 
            # Ellers appender vi nye dataframes i forlængelse af de gamle
            if os.path.exists(os.path.join(path_to_processed_folder,filename_person)):
                os.remove(os.path.join(path_to_processed_folder,filename_person))
            if not os.path.isdir(person_sti): continue
            person_nummer +=1
            
            # 3. Loop: Runder
            for runde in sorted(os.listdir(person_sti)):
                runde_sti = os.path.join(person_sti, runde)
                if not os.path.isdir(runde_sti): continue
                
                # 4. Loop: Phaser
                for phase in sorted(os.listdir(runde_sti)):
                    phase_sti = os.path.join(runde_sti, phase)
                    #print(phase_sti)
                    if not os.path.isdir(phase_sti): continue
                    phase_df = create_phase_dataframe(phase_sti)
                    append_to_csv(phase_df, filename_person,path_to_processed_folder)
        print('Finished person ', person_nummer)
                    

create_collected_datafile_all_participans()

Finished person  9
Finished person  15
Finished person  19
Finished person  23
Finished person  25
Finished person  27


In [41]:

df1 = pd.read_csv('/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/preprocessed/Person1_D1_1_ID_1.csv')
print(df1.columns)
print(len(df1))


Index(['time', 'BVP', 'EDA', 'HR', 'TEMP'], dtype='str')
64492


In [45]:
print(len(np.unique(df1['time'])))

64492


In [22]:
# import pandas as pd
# import pathlib
# import os

# # Definer stien
# dataset_path = r'/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset'
# path_to_folder = '../data/preprocessed'



# person_nummer = 1
# for gruppe in sorted(os.listdir(dataset_path)):
#     gruppe_sti = os.path.join(dataset_path, gruppe)
#     if not os.path.isdir(gruppe_sti): continue
    
#     # 2. Loop: Personer
#     for person in sorted(os.listdir(gruppe_sti)):
#         person_sti = os.path.join(gruppe_sti, person)
#         person_ID_and_group = f'Person{str(person_nummer)}_{gruppe}_{person}'
#         person_nummer +=1
#         filename_person = person_ID_and_group+'.csv'
#         #hvis koden er kørt før skal personfilerne slettes og laves på ny. 
#         # Ellers appender vi nye dataframes i forlængelse af de gamle
#         if os.path.exists(filename_person):
#             os.remove(filename_person)
        
#         if not os.path.isdir(person_sti): continue
        
#         # 3. Loop: Runder
#         for runde in sorted(os.listdir(person_sti)):
#             runde_sti = os.path.join(person_sti, runde)
#             if not os.path.isdir(runde_sti): continue
            
#             # 4. Loop: Phaser
#             for phase in sorted(os.listdir(runde_sti)):
#                 phase_sti = os.path.join(runde_sti, phase)
#                 #print(phase_sti)
#                 if not os.path.isdir(phase_sti): continue
#                 phase_df = create_phase_dataframe(phase_sti)
#                 append_to_csv(phase_df, filename_person,path_to_folder)
#     print('Finished person ', person_nummer)
                

Person1_D1_1_ID_1
/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1/round_1
Added 7153 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
Added 6257 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
Added 5905 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1/round_2
Added 4881 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
Added 5073 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
Added 5345 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset/D1_1/ID_1/round_3
Added 4737 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
Added 5025 rows to ../data/preprocessed/Person1_D1_1_ID_1.csv (Header added: False)
Added 4

In [ ]:
dataset_path = r'/Users/mathildenielsen/Desktop/02582_CDA/Case2/biosignal-analysis/data/raw/dataset'
Days = os.listdir(dataset_path)
print(Days)





for i, Day in enumerate(os.listdir(dataset_path)):
    print(i,Day)

['D1_1', 'D1_6', 'D1_3', 'D1_4', 'D1_5', 'D1_2']
0 D1_1
1 D1_6
2 D1_3
3 D1_4
4 D1_5
5 D1_2


In [181]:
os.listdir(dataset_path)

['D1_1', 'D1_6', 'D1_3', 'D1_4', 'D1_5', 'D1_2']

In [174]:
dict1 = {}

for i in range(10):
    dict1[f"Agens phase {i}"] = 10


dict1

{'Agens phase 0': 10,
 'Agens phase 1': 10,
 'Agens phase 2': 10,
 'Agens phase 3': 10,
 'Agens phase 4': 10,
 'Agens phase 5': 10,
 'Agens phase 6': 10,
 'Agens phase 7': 10,
 'Agens phase 8': 10,
 'Agens phase 9': 10}